In [1]:
!pip install -q transformers sentence-transformers datasets accelerate bitsandbytes openpyxl
!pip install -q langchain langchain_community langchain_core langchain_classic
!pip install -q langchain_huggingface langchain_ollama huggingface_hub faiss-cpu
!pip install -q fastapi uvicorn streamlit requests pyngrok pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 84.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 128.8 MB/s eta 0:00:0000:01


In [2]:
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install langchain-ollama

import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

!ollama pull llama3.2:3B

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils
0 upgraded, 3 newly installed, 0 to remove and 42 not upgraded.
Need to get 343 kB of archives.
After this operation, 1,581 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Fetched 343 kB in 1s (287 kB/s)  
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend: Readline
debconf: 

In [3]:
!ollama pull llama3.2:3B

In [4]:
import os
import sys

# Make local src files importable in Colab
candidate_dirs = [
    "./src",
    "/content/src",
    "/content/LLM_Project/src",
    "/content/drive/MyDrive/LLM_Project/src",
]
for path in candidate_dirs:
    if path not in sys.path and os.path.exists(path):
        sys.path.insert(0, path)

# Shared model/prompt pipeline + shared guardrails module
import llm_llama
from guardrails import (
    default_guardrails,
    is_clearly_out_of_scope,
    has_sufficient_context,
    contains_unsupported_numbers,
    contains_sensitive_data,
    looks_like_prompt_injection,
)

llm_llama.refresh_rag_resources()


def guardrailed_answer(query: str, k: int = 5) -> str:
    if looks_like_prompt_injection(query, default_guardrails):
        return default_guardrails.prompt_injection_reply

    docs = llm_llama.vec_db.similarity_search(query, k=k)
    contexts = [d.page_content for d in docs]

    if not has_sufficient_context(contexts, default_guardrails):
        return default_guardrails.insufficient_context_reply

    if is_clearly_out_of_scope(query, contexts, default_guardrails):
        return default_guardrails.out_of_domain_reply

    context_text = "\n\n".join(contexts)
    result = llm_llama.qa_chain.invoke({"query": query})
    answer = result["result"] if isinstance(result, dict) and "result" in result else str(result)

    if contains_unsupported_numbers(answer, context_text):
        print("[guardrail warning] Answer may include numbers not explicitly present in retrieved context.")

    if contains_sensitive_data(answer, default_guardrails):
        return default_guardrails.sensitive_output_reply

    return answer

print("Guardrails + model loaded from src. Use guardrailed_answer('your question').")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Guardrails + model loaded from src. Use guardrailed_answer('your question').


In [5]:
import time
import textwrap

benchmark_query = "Provide me more information regarding NUST Freelancer Digital account."

start_time = time.perf_counter()
benchmark_answer = guardrailed_answer(benchmark_query)
elapsed_seconds = time.perf_counter() - start_time

print("Benchmark query:", benchmark_query)
print(f"Inference time (guardrailed_answer): {elapsed_seconds:.2f} seconds")
print("\nAnswer:")
print(textwrap.fill(benchmark_answer, width=100))

[guardrail warning] Answer may include numbers not explicitly present in retrieved context.
Benchmark query: Provide me more information regarding NUST Freelancer Digital account.
Inference time (guardrailed_answer): 89.19 seconds

Answer:
I can provide more information about the NUST Freelancer Digital Account.   The NUST Freelancer
Digital Account is designed to cater to the needs of freelancers and independent contractors who
receive payments in various currencies, including PKR, USD, GBP, EUR, and AED. This account offers a
range of benefits, including:  *   Free issuance of first cheque book (25 leaves) *   Free first
issuance of Visa Debit Card *   Free first cheque book & debit card delivery *   Free SMS alerts
facility (digital transactions) *   Free accidental death & disability insurance/takaful up to PKR
2.5 mn on monthly average balance of PKR 100,000  The account type for NUST Freelancer Digital
Account is Current Account in PKR and FCY (USD, GBP, EUR, AED).  To open a NUS

In [6]:
!ollama ps

NAME           ID              SIZE      PROCESSOR    CONTEXT    UNTIL               
llama3.2:3B    a80c4f17acd5    2.8 GB    100% GPU     4096       29 minutes from now    


In [7]:
!nvidia-smi

Tue Apr 14 22:02:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P0             34W /   70W |    3137MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
import os
import time
import subprocess
import requests

!pip install -q fastapi uvicorn streamlit requests pyngrok python-dotenv

# Resolve project root (must contain src/api_server.py).
workspace_candidates = [
    "/content",
    "/content/LLM_Project",
    "/content/drive/MyDrive/LLM_Project",
]
project_root = next(
    (p for p in workspace_candidates if os.path.exists(os.path.join(p, "src", "api_server.py"))),
    None,
 )
if project_root is None:
    raise FileNotFoundError("src/api_server.py not found. Upload/sync project files to Colab first.")

src_dir = os.path.join(project_root, "src")
if not os.path.exists(os.path.join(src_dir, "streamlit_app.py")):
    raise FileNotFoundError("src/streamlit_app.py not found in the same project folder.")

print(f"Using project root: {project_root}")
print(f"Using src directory: {src_dir}")

# Load .env from src first, then project root (if present)
from dotenv import load_dotenv

env_candidates = [
    os.path.join(src_dir, ".env"),
    os.path.join(project_root, ".env"),
]
env_path = next((p for p in env_candidates if os.path.exists(p)), None)
if env_path:
    load_dotenv(env_path, override=False)
    print(f"Loaded env vars from: {env_path}")
else:
    print("No .env file found in src or project directory; relying on runtime env vars.")

# Restart API if already running
if "api_proc" in globals() and api_proc is not None and api_proc.poll() is None:
    api_proc.terminate()
    time.sleep(1)

# Start API from project root so imports like `from src...` resolve correctly.
api_proc = subprocess.Popen(
    ["uvicorn", "src.api_server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=project_root,
)

# Wait for API health
api_ready = False
for _ in range(20):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).ok:
            api_ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(1)

if not api_ready:
    raise RuntimeError("API failed to start. Check src/api_server.py and dependencies.")

# Restart Streamlit if already running
if "st_proc" in globals() and st_proc is not None and st_proc.poll() is None:
    st_proc.terminate()
    time.sleep(1)

# Start Streamlit UI
st_proc = subprocess.Popen(
    [
        "streamlit", "run", "src/streamlit_app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--browser.serverAddress", "0.0.0.0",
    ],
    cwd=project_root,
)
time.sleep(5)

from pyngrok import ngrok

token = os.environ.get("NGROK_AUTHTOKEN") or os.environ.get("NGROK_TOKEN")
if not token:
    raise RuntimeError(
        "NGROK token is missing. Put it in .env (NGROK_AUTHTOKEN=... or NGROK_TOKEN=...) or run: %env NGROK_AUTHTOKEN=your_token_here"
    )
ngrok.set_auth_token(token)

if "ngrok_tunnel" in globals() and ngrok_tunnel is not None:
    try:
        ngrok.disconnect(ngrok_tunnel.public_url)
    except Exception:
        pass

ngrok_tunnel = ngrok.connect(8501)
print("Open this app URL:", ngrok_tunnel.public_url)
print("In Streamlit sidebar, API URL should be: http://127.0.0.1:8000/query")

Using project root: /content
Using src directory: /content/src
Loaded env vars from: /content/.env
Open this app URL: https://phenomenally-slavish-lilly.ngrok-free.dev                                
In Streamlit sidebar, API URL should be: http://127.0.0.1:8000/query


In [10]:
print("Not failed")

Not failed
